In [ ]:
import sys
import community as louvain_community
import networkx as nx
import itertools
import infomap
from collections import Counter

sys.path.append("../../legal-data-clustering/")
%run '../../legal-data-clustering/legal_data_clustering/pipeline/cd_cluster.py'
%run 'common.py'

import altair.vega.v5 as alt
import cdlib
import numpy as np
from cdlib.readwrite import read_community_json

# Utils

In [ ]:
def vega_circle_dendrogram_schema(data, default_size_filter, layout="tidy", greyscale=False, rotation_angle=0, default_radius=290):
    return {
      "$schema": "https://vega.github.io/schema/vega/v5.json",
      "width": 720,
      "height": 720,
      "padding": 5,
      "autosize": "none",
      "signals": [
        {
          "name": "radius", "value": default_radius,
          "bind": {"input": "range", "min": 200, "max": 300}
        },
        {
          "name": "min_size", "value": default_size_filter,
          "bind": {"input": "range", "min": default_size_filter, "max": default_size_filter*10}
        },
        {
          "name": "layout", "value": layout,
          "bind": {"input": "radio", "options": ["tidy", "cluster"]}
        },
        {
          "name": "links", "value": "line",
          "bind": {
            "input": "select",
            "options": ["line", "curve", "diagonal", "orthogonal"]
          }
        },
        { "name": "originX", "update": "width / 2" },
        { "name": "originY", "update": "height / 2" },
        { "name": "separation", "value": False, "bind": {"input": "checkbox"} }
      ],

      "data": [
        {
          "name": "tree",
          "values": data,
          "transform": [
            {
              "type": "filter",
              "expr": "datum.size > min_size",
            },
            {
              "type": "stratify",
              "key": "id",
              "parentKey": "parent"
            },
            {
              "type": "tree",
              "method": {"signal": "layout"},
              "size": [1, {"signal": "radius"}],
              "separation": {"signal": "separation"},
              "as": ["alpha", "radius", "depth", "children"]
            },
            {
              "type": "formula",
              "expr": f"(360 * datum.alpha + 270) % 360 + {rotation_angle}",
              "as":   "angle"
            },
            {
              "type": "formula",
              "expr": "PI * datum.angle / 180",
              "as":   "radians"
            },
            {
              "type": "formula",
              "expr": "inrange(datum.angle, [90, 270])",
              "as":   "leftside"
            },
            {
              "type": "formula",
              "expr": "originX + datum.radius * cos(datum.radians)",
              "as":   "x"
            },
            {
              "type": "formula",
              "expr": "originY + datum.radius * sin(datum.radians)",
              "as":   "y"
            }
          ]
        },
        {
          "name": "links",
          "source": "tree",
          "transform": [
            { "type": "treelinks" },
            {
              "type": "linkpath",
              "shape": {"signal": "links"}, "orient": "radial",
              "sourceX": "source.radians", "sourceY": "source.radius",
              "targetX": "target.radians", "targetY": "target.radius"
            }
          ]
        }
      ],

      "scales": [
        {
          "name": "color",
          "type": "linear",
          "range": {"scheme": "greys" if greyscale else "magma"},
          "domain": {"data": "tree", "field": "depth"},
          "zero": True
        }
      ],

      "marks": [
        {
          "type": "path",
          "from": {"data": "links"},
          "encode": {
            "update": {
              "x": {"signal": "originX"},
              "y": {"signal": "originY"},
              "path": {"field": "path"},
              "stroke": {"value": "#000"}
            }
          }
        },
        {
          "type": "symbol",
          "from": {"data": "tree"},
          "encode": {
            "enter": {
              "size": {"value": 100},
              "stroke": {"value": "#fff"}
            },
            "update": {
              "x": {"field": "x"},
              "y": {"field": "y"},
              "fill": {"scale": "color", "field": "depth"}
            }
          }
        },
        {
          "type": "text",
          "from": {"data": "tree"},
          "encode": {
            "enter": {
              "text": {"field": "name"},
              "fontSize": {"value": 10.8},
              "baseline": {"value": "middle"},
              "font": {"value": "Times New Roman"},
              "fontWeight": {"value": "bold"}
            },
            "update": {
              "x": {"field": "x"},
              "y": {"field": "y"},
              "dx": {"signal": "(datum.leftside ? -1 : 1) * 6"},
              "angle": {"signal": "datum.leftside ? datum.angle - 180 : datum.angle"},
              "align": {"signal": "datum.leftside ? 'right' : 'left'"},
            }
          }
        }
      ]
    }

In [ ]:
def name_row(D, data, abk, parent_level, name):
    nodes = [n for n in D.nodes if f'_{abk}_' in n]
    for node in nodes:
        for _ in range(parent_level):
            node = list(D.predecessors(node))[0]

        for row in data:
            if row['id'] == node:
                row['name'] = name

# Vergleich der Algorithmen für Gesetze und Bücher

In [ ]:
G_de_2019_louvain = nx.read_gpickle(
    "../../legal-networks-data/de/11_cluster_results/2019-01-01_0-0_1-0_-1_a-louvain_m1-0_s0.gpickle.gz"
)
community_de_2019_louvain = read_community_json(
    "../../legal-networks-data/de/11_cluster_results/2019-01-01_0-0_1-0_-1_a-louvain_m1-0_s0.json"
)
G_de_2019_infomap = nx.read_gpickle(
    "../../legal-networks-data/de/11_cluster_results/2019-01-01_0-0_1-0_-1_a-infomap_m1-0_s0.gpickle.gz"
)
community_de_2019_infomap = read_community_json(
    "../../legal-networks-data/de/11_cluster_results/2019-01-01_0-0_1-0_-1_a-infomap_m1-0_s0.json"
)
G_us_2019_louvain = nx.read_gpickle(
    "../../legal-networks-data/us/11_cluster_results/2019_0-0_1-0_-1_a-louvain_m1-0_s0.gpickle.gz"
)
community_us_2019_louvain = read_community_json(
    "../../legal-networks-data/us/11_cluster_results/2019_0-0_1-0_-1_a-louvain_m1-0_s0.json"
)
G_us_2019_infomap = nx.read_gpickle(
    "../../legal-networks-data/us/11_cluster_results/2019_0-0_1-0_-1_a-infomap_m1-0_s0.gpickle.gz"
)
community_us_2019_infomap = read_community_json(
    "../../legal-networks-data/us/11_cluster_results/2019_0-0_1-0_-1_a-infomap_m1-0_s0.json"
)

In [ ]:
def abbreviate_buch(elem):
    if elem.get("name"):
        elem = elem.copy()
        elem["name"] = elem["name"] \
            .replace("Buch", "B.") \
            .replace("Erstes ", "1. ") \
            .replace("Zweites ", "2. ") \
            .replace("Drittes ", "3. ") \
            .replace("Viertes ", "4. ") \
            .replace("Fünftes ", "5. ")
        elem["name"] = truncate(elem["name"], 20)
    return elem

## Louvain DE

In [ ]:
data = graph_to_vega_data(G_de_2019_louvain, 'de', min_size=9000, size_attr='tokens_n', truncate_len=999)

data = [abbreviate_buch(elem) for elem in data]

D = G_de_2019_louvain
# name_row(D, data, 'SGB-3', 1, '[Sozialrecht]')
# name_row(D, data, 'EStG', 2, '[Steuerrecht]')
# name_row(D, data, 'VwGO', 2, 'Verwaltungsrecht (insb. Umweltrecht)')
# name_row(D, data, 'GBO', 2, 'Zivilrecht')
# name_row(D, data, 'GmbHG', 2, '[Gesellschafts- und Kapitelmarktrecht]')
# name_row(D, data, 'ArbGG', 2, 'Bes. Zivilrecht')
# name_row(D, data, 'PStG', 1, 'Familienrecht')
# name_row(D, data, 'DRiG', 2, 'Beamtenrecht')
# name_row(D, data, 'BHO', 1, 'Haushaltsrecht')
# name_row(D, data, 'GG', 1, 'Verfassungsrecht')
# name_row(D, data, 'MessbG', 1, 'Datenschutzrecht')
# name_row(D, data, 'StGB', 1, 'Strafrecht')
# name_row(D, data, 'GewO', 2, 'Gewerberecht')
# name_row(D, data, 'AZRG', 1, 'Ausländerrecht')
# name_row(D, data, 'PatG', 1, 'Gew. Rechtsschutz')

chart = alt.Vega(vega_circle_dendrogram_schema(data, 9000, rotation_angle=7, default_radius=280))
used_abks = {d['name'].split(',')[0] for d in data if d.get('name')}
save_chart_and_crop(chart, 'meso_dendrogram_louvain_de_2019', used_abks)

In [ ]:
chart = alt.Vega(vega_circle_dendrogram_schema(data, 9000, greyscale=True, rotation_angle=7, default_radius=280))
save_chart_and_crop(chart, 'meso_dendrogram_louvain_de_2019_graycolor', used_abks)

## Infomap DE

In [ ]:
data = graph_to_vega_data(G_de_2019_infomap, 'de', min_size=9000, size_attr='tokens_n', truncate_len=999)
data = [abbreviate_buch(elem) for elem in data]

chart = alt.Vega(vega_circle_dendrogram_schema(data, 9000, layout='cluster', rotation_angle=0, default_radius=275))
used_abks = {d['name'].split(',')[0] for d in data if d.get('name')}
save_chart_and_crop(chart, 'meso_dendrogram_infomap_de_2019', used_abks)

In [ ]:
chart = alt.Vega(vega_circle_dendrogram_schema(data, 9000, layout='cluster', greyscale=True, rotation_angle=0, default_radius=275))
save_chart_and_crop(chart, 'meso_dendrogram_infomap_de_2019_graycolor', used_abks)

## Vergleich von Infomap und Louvain DE

In [ ]:
score = cdlib.evaluation.normalized_mutual_information(
    community_de_2019_louvain,community_de_2019_infomap
).score
diss_data(
    'meso_crossreference_2019_de_gesetze_infomap_louvain_compare_normalized_mutual_information',
    de_num_format(f'{score:.3f}')
)

score = cdlib.evaluation.adjusted_rand_index(
    community_de_2019_louvain,community_de_2019_infomap
).score
diss_data(
    'meso_crossreference_2019_de_gesetze_infomap_louvain_compare_adjusted_rand_index',
    de_num_format(f'{score:.3f}')
)

## Louvain US

In [ ]:
data = graph_to_vega_data(G_us_2019_louvain, 'us', min_size=9000, size_attr='tokens_n')
chart = alt.Vega(vega_circle_dendrogram_schema(data, 9000))
save_chart_and_crop(chart, 'meso_dendrogram_louvain_us_2019')

In [ ]:
chart = alt.Vega(vega_circle_dendrogram_schema(data, 9000, greyscale=True))
save_chart_and_crop(chart, 'meso_dendrogram_louvain_us_2019_graycolor')

## Infomap US

In [ ]:
data = graph_to_vega_data(G_us_2019_infomap, 'us', min_size=9000, size_attr='tokens_n')
chart = alt.Vega(vega_circle_dendrogram_schema(data, 9000, layout='cluster'))
save_chart_and_crop(chart, 'meso_dendrogram_infomap_us_2019')

In [ ]:
chart = alt.Vega(vega_circle_dendrogram_schema(data, 9000, layout='cluster', greyscale=True))
save_chart_and_crop(chart, 'meso_dendrogram_infomap_us_2019_grayscolor')

## Vergleich von Infomap und Louvain US

In [ ]:
score = cdlib.evaluation.normalized_mutual_information(
    community_us_2019_louvain,community_us_2019_infomap
).score
diss_data(
    'meso_crossreference_2019_us_gesetze_infomap_louvain_compare_normalized_mutual_information',
    de_num_format(f'{score:.3f}')
)

score = cdlib.evaluation.adjusted_rand_index(
    community_us_2019_louvain,community_us_2019_infomap
).score
diss_data(
    'meso_crossreference_2019_us_gesetze_infomap_louvain_compare_adjusted_rand_index',
    de_num_format(f'{score:.3f}')
)